In [115]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import os

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

appointments = pd.read_sql("SELECT * FROM appointments", engine)
providers = pd.read_sql("SELECT provider_id, specialty, primary_location_id FROM providers", engine)

print(appointments.shape)
appointments.head()

(199420, 12)


,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2022-07-20,2022-07-20,P2,L1,PT1,PY1,Follow-up,False,Completed,125.50,0.81
1,A2,2022-07-20,2022-07-19,P2,L1,PT1,PY1,Follow-up,False,Completed,75.40,0.83
2,A3,2022-07-20,2022-07-02,P2,L1,PT2,PY3,New Patient Consult,True,Completed,258.34,2.47
3,A4,2022-07-20,2022-07-03,P2,L1,PT2,PY3,Follow-up,False,Completed,163.69,1.15
4,A5,2022-07-20,2022-07-06,P2,L1,PT3,PY5,New Patient Consult,True,Completed,297.58,1.97


In [116]:
df = appointments[appointments['status'].isin(['Completed', 'No-Show'])].copy()
df['no_show'] = (df['status'] == 'No-Show').astype(int)
print(df['no_show'].value_counts())
print(df['no_show'].mean())  # what % are no-shows overall

no_show
0    165110
1     24168
Name: count, dtype: int64
0.1276852037743425


In [117]:
df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu,no_show
0,A1,2022-07-20,2022-07-20,P2,L1,PT1,PY1,Follow-up,False,Completed,125.50,0.81,0
1,A2,2022-07-20,2022-07-19,P2,L1,PT1,PY1,Follow-up,False,Completed,75.40,0.83,0
2,A3,2022-07-20,2022-07-02,P2,L1,PT2,PY3,New Patient Consult,True,Completed,258.34,2.47,0
3,A4,2022-07-20,2022-07-03,P2,L1,PT2,PY3,Follow-up,False,Completed,163.69,1.15,0
4,A5,2022-07-20,2022-07-06,P2,L1,PT3,PY5,New Patient Consult,True,Completed,297.58,1.97,0


In [118]:
df.info()

<class 'pandas.DataFrame'>
Index: 189278 entries, 0 to 199419
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   appointment_id    189278 non-null  str    
 1   date              189278 non-null  object 
 2   booked_date       188890 non-null  object 
 3   provider_id       189278 non-null  str    
 4   location_id       189278 non-null  str    
 5   patient_id        189278 non-null  str    
 6   payer_id          189278 non-null  str    
 7   appointment_type  189278 non-null  str    
 8   is_new_patient    189278 non-null  bool   
 9   status            189278 non-null  str    
 10  revenue           187629 non-null  float64
 11  rvu               189278 non-null  float64
 12  no_show           189278 non-null  int64  
dtypes: bool(1), float64(2), int64(1), object(2), str(7)
memory usage: 19.0+ MB


In [119]:
# Convert dates
df['date'] = pd.to_datetime(df['date'], errors='coerce', format='%Y-%m-%d')
df['booked_date'] = pd.to_datetime(df['booked_date'], errors='coerce', format='%Y-%m-%d')

# Likely strongest predictor, days ahead of when booked
df['lead_time_days'] = (df['date'] - df['booked_date']).dt.days

# Null times do not contribute, drop impossible dates set to null in data cleaning for training purposes
print(f"Null lead times: {df['lead_time_days'].isna().sum()}")
df = df.dropna(subset=['booked_date'])

# 0 = Monday, Sundays are always closed
df['day_of_week'] = df['date'].dt.day_of_week
print(df.groupby('day_of_week')['no_show'].mean())

# For seasonality perhaps
df['month'] = df['date'].dt.month

# Merge provider specialty and location
df = df.merge(providers, on='provider_id', how='left')

Null lead times: 388
day_of_week
0    0.126107
1    0.127808
2    0.129097
3    0.129823
4    0.125668
5    0.128430
Name: no_show, dtype: float64


In [120]:
# Does lead time actually correlate with no-show rate?
# Too little data for 30-999?
df.groupby(pd.cut(df['lead_time_days'], bins=[0,3,7,14,30,999]))['no_show'].mean()

lead_time_days
(0, 3]       0.115492
(3, 7]       0.118687
(7, 14]      0.133906
(14, 30]     0.165993
(30, 999]    0.000000
Name: no_show, dtype: float64

In [121]:
# Value verification
print(df['lead_time_days'].value_counts().sort_index().tail(30))
print(df[df['lead_time_days'].between(14, 30)].shape)

lead_time_days
2.0      9025
3.0     10532
4.0     12075
5.0     12950
6.0     13783
7.0     13885
8.0     13666
9.0     13075
10.0    11821
11.0    10407
12.0     9058
13.0     7627
14.0     6135
15.0     4890
16.0     3664
17.0     2666
18.0     1938
19.0     1329
20.0      884
21.0      615
22.0      349
23.0      229
24.0      109
25.0       75
26.0       29
27.0       26
28.0       11
29.0        4
30.0        2
31.0        1
Name: count, dtype: int64
(22955, 18)


In [122]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 188890 entries, 0 to 188889
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype        
---  ------               --------------   -----        
 0   appointment_id       188890 non-null  str          
 1   date                 188890 non-null  datetime64[s]
 2   booked_date          188890 non-null  datetime64[s]
 3   provider_id          188890 non-null  str          
 4   location_id          188890 non-null  str          
 5   patient_id           188890 non-null  str          
 6   payer_id             188890 non-null  str          
 7   appointment_type     188890 non-null  str          
 8   is_new_patient       188890 non-null  bool         
 9   status               188890 non-null  str          
 10  revenue              187243 non-null  float64      
 11  rvu                  188890 non-null  float64      
 12  no_show              188890 non-null  int64        
 13  lead_time_days       188890 non-null  fl

In [123]:
num_features = ['lead_time_days']
cat_features = ['day_of_week', 'month', 'is_new_patient', 'appointment_type', 'specialty', 'location_id', 'payer_id', 'provider_id']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

In [124]:
# Make df
feature_cols = num_features + cat_features

X = df[feature_cols].copy()
y = df['no_show']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train no-show rate: {y_train.mean():.3f}")
print(f"Test no-show rate:  {y_test.mean():.3f}")

Train: (151112, 9), Test: (37778, 9)
Train no-show rate: 0.128
Test no-show rate:  0.128


In [125]:
# Train and predict
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Encoded results
encoded_cols = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder'].get_feature_names_out()
print(encoded_cols)

['x0_0' 'x0_1' 'x0_2' 'x0_3' 'x0_4' 'x0_5' 'x1_1' 'x1_2' 'x1_3' 'x1_4'
 'x1_5' 'x1_6' 'x1_7' 'x1_8' 'x1_9' 'x1_10' 'x1_11' 'x1_12' 'x2_False'
 'x2_True' 'x3_Follow-up' 'x3_Injection/Procedure'
 'x3_New Patient Consult' 'x3_Physical Therapy' 'x3_Post-Op Check'
 'x4_Hand & Upper Extremity' 'x4_Orthopedic Surgery' 'x4_Pain Management'
 'x4_Physical Medicine & Rehab' 'x4_Physical Therapy' 'x4_Spine Surgery'
 'x4_Sports Medicine' 'x4_Unknown' 'x5_L1' 'x5_L2' 'x5_L3' 'x5_L4' 'x5_L5'
 'x5_L6' 'x5_L7' 'x6_PY1' 'x6_PY2' 'x6_PY3' 'x6_PY4' 'x6_PY5' 'x6_PY6'
 'x6_PY7' 'x7_P1' 'x7_P10' 'x7_P11' 'x7_P12' 'x7_P13' 'x7_P14' 'x7_P15'
 'x7_P16' 'x7_P17' 'x7_P18' 'x7_P19' 'x7_P2' 'x7_P20' 'x7_P21' 'x7_P22'
 'x7_P23' 'x7_P24' 'x7_P25' 'x7_P26' 'x7_P27' 'x7_P3' 'x7_P4' 'x7_P5'
 'x7_P6' 'x7_P7' 'x7_P8' 'x7_P9' 'x7_UNK']


In [126]:
# Evaluate
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred, target_names=['Completed', 'No-Show']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.6409183848374164
              precision    recall  f1-score   support

   Completed       0.91      0.59      0.72     32951
     No-Show       0.18      0.62      0.28      4827

    accuracy                           0.59     37778
   macro avg       0.55      0.60      0.50     37778
weighted avg       0.82      0.59      0.66     37778

Confusion Matrix:
[[19378 13573]
 [ 1838  2989]]


In [127]:
# Can opt for a lower threshold to catch more no-shows depending on the cost of acting upon it
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print(classification_report(
        y_test,
        y_pred_threshold,
        target_names=['Completed', 'No-Show'],
        zero_division=0
    ))


Threshold: 0.2
              precision    recall  f1-score   support

   Completed       0.00      0.00      0.00     32951
     No-Show       0.13      1.00      0.23      4827

    accuracy                           0.13     37778
   macro avg       0.06      0.50      0.11     37778
weighted avg       0.02      0.13      0.03     37778


Threshold: 0.3
              precision    recall  f1-score   support

   Completed       0.95      0.08      0.15     32951
     No-Show       0.13      0.97      0.24      4827

    accuracy                           0.19     37778
   macro avg       0.54      0.52      0.19     37778
weighted avg       0.84      0.19      0.16     37778


Threshold: 0.4
              precision    recall  f1-score   support

   Completed       0.94      0.34      0.50     32951
     No-Show       0.16      0.84      0.26      4827

    accuracy                           0.40     37778
   macro avg       0.55      0.59      0.38     37778
weighted avg       0.84   

In [ ]:
# RandomForest test
pipeline2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

pipeline2.fit(X_train, y_train)

y_pred2 = pipeline2.predict(X_test)
y_prob2 = pipeline2.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob2))
print(classification_report(y_test, y_pred2, target_names=['Completed', 'No-Show']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred2))

['x0_0' 'x0_1' 'x0_2' 'x0_3' 'x0_4' 'x0_5' 'x1_1' 'x1_2' 'x1_3' 'x1_4'
 'x1_5' 'x1_6' 'x1_7' 'x1_8' 'x1_9' 'x1_10' 'x1_11' 'x1_12' 'x2_False'
 'x2_True' 'x3_Follow-up' 'x3_Injection/Procedure'
 'x3_New Patient Consult' 'x3_Physical Therapy' 'x3_Post-Op Check'
 'x4_Hand & Upper Extremity' 'x4_Orthopedic Surgery' 'x4_Pain Management'
 'x4_Physical Medicine & Rehab' 'x4_Physical Therapy' 'x4_Spine Surgery'
 'x4_Sports Medicine' 'x4_Unknown' 'x5_L1' 'x5_L2' 'x5_L3' 'x5_L4' 'x5_L5'
 'x5_L6' 'x5_L7' 'x6_PY1' 'x6_PY2' 'x6_PY3' 'x6_PY4' 'x6_PY5' 'x6_PY6'
 'x6_PY7' 'x7_P1' 'x7_P10' 'x7_P11' 'x7_P12' 'x7_P13' 'x7_P14' 'x7_P15'
 'x7_P16' 'x7_P17' 'x7_P18' 'x7_P19' 'x7_P2' 'x7_P20' 'x7_P21' 'x7_P22'
 'x7_P23' 'x7_P24' 'x7_P25' 'x7_P26' 'x7_P27' 'x7_P3' 'x7_P4' 'x7_P5'
 'x7_P6' 'x7_P7' 'x7_P8' 'x7_P9' 'x7_UNK']
ROC-AUC: 0.5727482823385097
              precision    recall  f1-score   support

   Completed       0.88      0.88      0.88     32951
     No-Show       0.16      0.16      0.16      4827

In [ ]:
# CalibratedClassifier test
base_clf = LogisticRegression(max_iter=1000, class_weight='balanced')

pipeline3 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', CalibratedClassifierCV(base_clf, cv=5, method="sigmoid"))
])

pipeline3.fit(X_train, y_train)

y_pred3 = pipeline3.predict(X_test)
y_prob3 = pipeline3.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob3))
print(classification_report(y_test, y_pred3, target_names=['Completed', 'No-Show'], zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred3))

['x0_0' 'x0_1' 'x0_2' 'x0_3' 'x0_4' 'x0_5' 'x1_1' 'x1_2' 'x1_3' 'x1_4'
 'x1_5' 'x1_6' 'x1_7' 'x1_8' 'x1_9' 'x1_10' 'x1_11' 'x1_12' 'x2_False'
 'x2_True' 'x3_Follow-up' 'x3_Injection/Procedure'
 'x3_New Patient Consult' 'x3_Physical Therapy' 'x3_Post-Op Check'
 'x4_Hand & Upper Extremity' 'x4_Orthopedic Surgery' 'x4_Pain Management'
 'x4_Physical Medicine & Rehab' 'x4_Physical Therapy' 'x4_Spine Surgery'
 'x4_Sports Medicine' 'x4_Unknown' 'x5_L1' 'x5_L2' 'x5_L3' 'x5_L4' 'x5_L5'
 'x5_L6' 'x5_L7' 'x6_PY1' 'x6_PY2' 'x6_PY3' 'x6_PY4' 'x6_PY5' 'x6_PY6'
 'x6_PY7' 'x7_P1' 'x7_P10' 'x7_P11' 'x7_P12' 'x7_P13' 'x7_P14' 'x7_P15'
 'x7_P16' 'x7_P17' 'x7_P18' 'x7_P19' 'x7_P2' 'x7_P20' 'x7_P21' 'x7_P22'
 'x7_P23' 'x7_P24' 'x7_P25' 'x7_P26' 'x7_P27' 'x7_P3' 'x7_P4' 'x7_P5'
 'x7_P6' 'x7_P7' 'x7_P8' 'x7_P9' 'x7_UNK']
ROC-AUC: 0.6410156345363356
              precision    recall  f1-score   support

   Completed       0.87      1.00      0.93     32951
     No-Show       0.00      0.00      0.00      4827

In [130]:
# Dummy verification
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

dummy_prob = dummy.predict_proba(X_test)[:, 1]

print(roc_auc_score(y_test, dummy_prob))

0.5


In [144]:
# Run on entire set
all_probs = pipeline.predict_proba(X)[:, 1]
all_preds = pipeline.predict(X)

# Build the predictions dataframe to write to Postgres
predictions_df = pd.DataFrame({
    'appointment_id': df['appointment_id'].values,
    'no_show_actual': y.values,
    'no_show_predicted': all_preds,
    'risk_score': all_probs,
    'risk_tier': pd.qcut(
        all_probs,
        q=[0, 0.65, 0.85, 1.0],
        labels=['Low', 'Medium', 'High']
    )
})

print(predictions_df.head())
print(predictions_df['risk_tier'].value_counts())
print(f"Avg predicted probability: {all_probs.mean():.3f}")

print("Actual no-show rate by tier:")
print(predictions_df.groupby('risk_tier', observed=True)['no_show_actual'].mean())

print("ROC-AUC:", roc_auc_score(y, all_probs))

  appointment_id  no_show_actual  no_show_predicted  risk_score risk_tier
0             A1               0                  0    0.387231       Low
1             A2               0                  0    0.389081       Low
2             A3               0                  1    0.542128    Medium
3             A4               0                  0    0.426385       Low
4             A5               0                  1    0.518081       Low
risk_tier
Low       122778
Medium     37778
High       28334
Name: count, dtype: int64
Avg predicted probability: 0.477
Actual no-show rate by tier:
risk_tier
Low       0.093339
Medium    0.168246
High      0.222948
Name: no_show_actual, dtype: float64
ROC-AUC: 0.643792952374449


In [142]:

predictions_df['risk_tier'] = predictions_df['risk_tier'].astype(str)

predictions_df.to_sql(
    'no_show_predictions',
    con=engine,
    if_exists='replace',
    index=False
)
print(f"Written {len(predictions_df)} rows to no_show_predictions")

Written 188890 rows to no_show_predictions
